In [4]:
import os
import numpy as np 
import pandas as pd 

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\aryan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

c:\Users\aryan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\aryan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-s

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [10]:
train_ds = ds["train"]
validation_ds = ds["validation"]

In [15]:
train_ds[:5]

{'text': ['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
  'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leav

In [ ]:
batch_size = 1000

In [17]:
def batch_iterator():
    for i in range(0, len(train_ds), batch_size):
        yield train_ds[i : i + batch_size]["text"]

In [18]:
bpe_tokenizer = tokenizer.train_new_from_iterator(batch_iterator(), vocab_size=4096)

In [19]:
len(bpe_tokenizer)

4096

In [20]:
bpe_tokenizer.encode("<|endoftext|>")  

[0]

In [22]:
text = train_ds[0]

In [27]:
encoded_text = bpe_tokenizer(text["text"])
input_ids = encoded_text["input_ids"]

In [28]:
decoded_text = bpe_tokenizer.decode(input_ids)
decoded_text

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [29]:
bpe_tokenizer.save_pretrained("tokenizer")

('tokenizer\\tokenizer_config.json', 'tokenizer\\tokenizer.json')

In [39]:
tok = AutoTokenizer.from_pretrained("tokenizer")
EOT_ID = tok.convert_tokens_to_ids("<|endoftext|>")
assert EOT_ID == 0
assert len(tok) < 2**16 

In [42]:
os.makedirs('data', exist_ok=True)

def tokenize(batch):
    ids = tok(batch["text"], add_special_tokens=False)["input_ids"]
    ids = [x + [EOT_ID] for x in ids]
    return {"ids" : ids, "len": [len(x) for x in ids]}

def write_bin(split_ds, path, num_chunks=1024):
    
    tokenized = split_ds.map(
        tokenize,
        batched=True,
        remove_columns=["text"],           
        desc=f"tokenizing -> {path}",
    )
    total = int(np.asarray(tokenized["len"], dtype=np.int64).sum())

    arr = np.memmap(path, dtype= np.uint16, mode = "w+", shape = (total,))

    idx = 0
    for i in range(num_chunks):
        chunk = tokenized.shard(num_shards = num_chunks, index = i, contiguous = True)
        chunk = chunk.with_format("numpy")["ids"]
        flat = np.concatenate(chunk).astype(np.uint16)
        arr[idx : idx + len(flat)] = flat
        idx += len(flat)
    arr.flush()
    assert idx == total, f"wrote {idx}, expected {total}"
    print(f"{path}: {total:,} tokens, {total * 2 / 1e6:.1f} MB")
    return total

n_val = write_bin(validation_ds, "data/val.bin")
# n_train = write_bin(train_ds, "data/train.bin")

tokenizing -> data/val.bin:   0%|          | 0/21990 [00:00<?, ? examples/s]

data/val.bin: 4,883,717 tokens, 9.8 MB


In [43]:
n_train = write_bin(train_ds, "data/train.bin")

tokenizing -> data/train.bin:   0%|          | 0/2119719 [00:00<?, ? examples/s]

data/train.bin: 485,795,451 tokens, 971.6 MB


In [44]:
m = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
tok.decode(m[:200].tolist())

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.<|endoftext|>Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because'